---
layout: post
types: hacks
title: Full Stack Review
description: A review of full stack!
toc: true
comments: true
---

# Intro

### Group Program
- Help students learn how to convert using binary
- Interactive games to keep students motivated while having fun
- Collaborative site for students 

### Individual Feature
- A player analytics table which stores a players name, id, games played, average score, wins, losses, and highest score
- Purpose: show others different players scores and data
- Motivate them to keep playing and do better

# Input/Output Requests


In [ ]:
async function readScores() {
    try {
      const response = await fetch(`${pythonURI}/api/firstPlaceLeaderboard`, fetchOptions);
  
      if (!response.ok) {
        throw new Error('Failed to fetch scores');
      }
  
      const scores = await response.json();
      const tableBody = document.getElementById('scoresTableBody');
      tableBody.innerHTML = ''; // Clear previous rows
  
      scores.forEach(score => {
        const row = document.createElement('tr');
        row.innerHTML = `
          <td>${score.username}</td>
          <td>${score.user_id}</td>
          <td>${score.user_games_played}</td>
          <td>${score.user_average_score}</td>
          <td>${score.user_wins}</td>
          <td>${score.user_losses}</td>
          <td>${score.user_highest_score}</td>
        `;
        tableBody.appendChild(row);
      });
  
    } catch (error) {
      console.error('Error fetching scores:', error);
    }
  }

This code fetches data from an API endpoint and dynamically updates the DOM to display the leaderboard. It includes error handling for failed requests.

- Gets scores from an API: Fetches data from a server using a link (pythonURI).
- Checks if the request worked: If it fails, an error message is shown.
- Clears the current table: Removes old scores from the page.
- Adds new rows to the table:
- Shows details like username, games played, wins, losses, and highest score.
- Handles errors: Logs any issues to the console.

### Using Postman for API Testing

<span>
<img src="{{site.baseurl}}images/postman.png" width = "500" height = "300"/>
</span> 

This is showing a POST request and the response provides the leaderboard data for a user named "cool_coder" who has played 6 games, achieved an average score of 8, won 2 games, lost 2, and reached a highest score of 8. The server successfully saved the user data and returned an object with additional properties.


### Tester Data Creation

In [ ]:
def initFirstPlaceLeaderboard():
    with app.app_context():
        """Create database and tables"""
        db.create_all()
        """Tester data for table"""

        format = "%Y-%m-%d %H:%M:%S"

        p1 = firstPlaceLeaderboard(username="JIM", user_id="jim_is_the_best", games_played="5", average_score="5.0", wins="3", losses="2", highest_score="10")
        p2 = firstPlaceLeaderboard(username="TIM", user_id="tim_10", games_played="3", average_score="3.0", wins="2", losses="1", highest_score="5")
        p3 = firstPlaceLeaderboard(username="BUM", user_id="dum_bum", games_played="7", average_score="4.0", wins="4", losses="3", highest_score="7")
        p4 = firstPlaceLeaderboard(username="TUM", user_id="tum123", games_played="4", average_score="4.5", wins="3", losses="1", highest_score="8")
        
        for post in [p1, p2, p3, p4]:
            try:
                post.create()
                print(f"Record created: {repr(post)}")
            except IntegrityError:
                '''fails with bad or duplicate data'''
                db.session.remove()
                print(f"Records exist, duplicate email, or error: {post.user_id}")

<span>
<img src="{{site.baseurl}}images/database.png" width = "900" height = "250"/>
</span> 

Explanation: 
- This function initializes the database schema and adds test data.
- ensures there is no duplicate data

# List Requests

### Formatting response data (JSON) from API into DOM

In [ ]:
async function createData(username, userId, gamesPlayed, averageScore, wins, losses, highestScore) {
  const scoreData = {
    username,
    user_id: userId,
    user_games_played: gamesPlayed,
    user_average_score: averageScore,
    user_wins: wins,
    user_losses: losses,
    user_highest_score: highestScore,
  };

  try {
    const response = await fetch(`${pythonURI}/api/firstPlaceLeaderboard`, {
      ...fetchOptions,
      method: 'POST',
      body: JSON.stringify(scoreData),
    });

    if (!response.ok) {
      throw new Error(`Failed to submit data: ${response.statusText}`);
    }

    const result = await response.json();
    console.log('Data submitted successfully:', result);

  } catch (error) {
    console.error('Error submitting data:', error);
    alert('Error submitting data: ' + error.message);
  }
}

- sends a request to a server to the URL pythonURI/api/firstPlaceLeaderboard.
- Uses POST to send data.
- Sends scoreData as JSON in the request body.
- if (!response.ok): Checks if the response from the server is successful.
- throw new Error(...): If the request fails, it throws an error with the failure message.

### Queries From Database

In [ ]:
def get(self):
# Find all the posts by the current user
posts = firstPlaceLeaderboard.query.all()
# Prepare a JSON list of all the posts, uses for loop shortcut called list comprehension
json_ready = [post.read() for post in posts]
# Return a JSON list, converting Python dictionaries to JSON format
return jsonify(json_ready)

- It fetches all posts from the database using SQLAlchemy and returns them as JSON using Flask.
- 3rd party library: SQLAlchemy
- SQLAlchemy: Used for interacting with the database and running the query.all() to retrieve all posts.

### Methods in "class" (create, read, update, delete)

In [ ]:
def __init__(self, username, user_id, games_played, average_score, wins, losses, highest_score):
    self._username = username
    self._user_id = user_id
    self._games_played = games_played
    self._average_score = average_score
    self._wins = wins
    self._losses = losses
    self._highest_score = highest_score

- init in model assigns values for the table’s columns

In [ ]:
class _CRUD(Resource):
        @token_required()
        def post(self):
            # Obtain the current user from the token
            current_user = g.current_user
            # Obtain the request data sent by the RESTful client API
            data = request.get_json()
            # Create a new post object using the data from the request
            post = firstPlaceLeaderboard(data['username'], current_user.id, data['games_played'], data['average_score'], data['wins'], data['losses'], data['highest_score'])
            format= "%Y-%m-%d %H:%M:%S"
            # Save the post object using the ORM method defined in the model
            post.create()

            return jsonify(post.read())

        def get(self):
            # Obtain the current user
            # current_user = g.current_user
            # Find all the posts by the current user
            posts = firstPlaceLeaderboard.query.all()
            # Prepare a JSON list of all the posts, uses for loop shortcut called list comprehension
            json_ready = [post.read() for post in posts]
            # Return a JSON list, converting Python dictionaries to JSON format
            return jsonify(json_ready)
        
        def put(self):
            # Obtain the request data
            data = request.get_json()
            # Find the current post from the database table(s)
            post = firstPlaceLeaderboard.query.get(data['id'])
            # Update the post using the ORM method defined in the model
            post.update(data)
            # Return response
            return jsonify(post.read())
        
        @token_required()
        def delete(self):
            # Obtain the current user
            current_user = g.current_user
            # Obtain the request data
            data = request.get_json()
            # Find the current post from the database table(s)
            post = firstPlaceLeaderboard.query.get(data['id'])
            # Delete the post using the ORM method defined in the model
            post.delete()
            # Return response
            return jsonify({"message": "Post deleted"})

1. post(self):
- Creates a new leaderboard post.
- Gets the current user and data sent by the client.
- Saves the new post to the database.
- Returns the created post as a JSON response.
2. get(self):
- Retrieves all leaderboard posts from the database.
- Converts the posts into a JSON list.
- Returns the list of posts in JSON format.
3. put(self):
- Updates an existing leaderboard post.
- Retrieves the post from the database using its ID.
- Updates the post with new data.
- Returns the updated post as a JSON response.
4. delete(self):
- Deletes a leaderboard post.
- Retrieves the post using its ID.
- Deletes the post from the database.
- Returns a message saying the post has been deleted.

# Algorithmic Code Request

### API Class

In [ ]:
def create(self):
        try:
            db.session.add(self)
            db.session.commit()
        except Exception as e:
            db.session.rollback()
            raise e
        
    def read(self):
        user = User.query.get(self._user_id)
        data = {
            "id": self.id,
            "username": self._username,
            "user_id": self._user_id if user else None,
            "user_games_played": self._games_played,
            "average_score": self._average_score,
            "wins": self._wins,
            "losses": self._losses,
            "highest_score": self._highest_score
        }
        return data
    
    def update(self, inputs):
        if not isinstance(inputs, dict):
            return self
        
        games_played = inputs.get('games_played')
        average_score = inputs.get('average_score')
        wins = inputs.get('wins')
        losses = inputs.get('losses')
        highest_score = inputs.get('highest_score')

        if (games_played):
            self._games_played = games_played
        if (average_score):
            self._average_score = average_score
        if (wins):
            self._wins = wins
        if (losses):
            self._losses = losses
        if (highest_score):
            self._highest_score = highest_score

        try:
            db.session.commit()
        except Exception as e:
            db.session.rollback()
            raise e
        return self
    
    def delete(self):  
        try:
            db.session.delete(self)
            db.session.commit()
        except Exception as e:
            db.session.rollback()
            raise e
        
    @staticmethod
    def restore(data):
        sections = {}
        existing_sections = {section._username: section for section in firstPlaceLeaderboard.query.all()}
        for section_data in data:
            _ = section_data.pop('id', None)  # Remove 'id' from section_data
            username = section_data.get("username", None)
            section = existing_sections.pop(username, None)
            print(section_data)
            if section:
                section.update(section_data)
            else:
                section = firstPlaceLeaderboard(**section_data)
                section.create()
        
        # Remove any extra data that is not in the backup
        for section in existing_sections.values():
            db.session.delete(section)
        
        db.session.commit()
        return sections

This code provides methods to create, read, update, delete, and restore leaderboard records in a database using SQLAlchemy. It handles errors by rolling back changes when necessary.

### Sequencing, Selection, Iteration

In [ ]:
def restore(data):
        sections = {}
        existing_sections = {section._username: section for section in firstPlaceLeaderboard.query.all()}
        for section_data in data:
            _ = section_data.pop('id', None)  # Remove 'id' from section_data
            username = section_data.get("username", None)
            section = existing_sections.pop(username, None)
            print(section_data)
            if section:
                section.update(section_data)
            else:
                section = firstPlaceLeaderboard(**section_data)
                section.create()
        

1. Sequencing
- code starts by creating empty dictionaries (sections, existing_sections) and fetching all existing sections from the database.
- loops through the provided data, removes the id, checks if a username exists, and either updates or creates a record.
2. Selection
- if section:: Checks if the section already exists in existing_sections.
- If True, it updates the section (section.update(section_data)).
- If False, it creates a new section using firstPlaceLeaderboard(**section_data) and saves it with section.create().
3. Iteration
- for section_data in data:: Loops through each section in the input data.
- Processes each item by updating or creating entries as needed.

### Body of Request

In [ ]:
{"username": "test", "games_played": 6, "average_score": 8, "wins": 2, "losses": 2, "highest_score": 8}

### Return Type

In [ ]:
{
    "average_score": 8,
    "highest_score": 8,
    "id": 5,
    "losses": 2,
    "user_games_played": 6,
    "user_id": "1",
    "username": "test",
    "wins": 2
}

# Call to Algorithm request


### Call to Method

In [ ]:
fetch(`${pythonURI}/api/firstPlaceLeaderboard`)
  .then(response => response.json())
  .then(data => console.log('Fetched Data:', data))
  .catch(error => console.error('Error:', error));


  try {
    const response = await fetch(`${pythonURI}/api/firstPlaceLeaderboard`, {
      ...fetchOptions,
      method: 'POST',
      body: JSON.stringify(scoreData),
    });

    if (!response.ok) {
      throw new Error(`Failed to submit data: ${response.statusText}`);
    }

    const result = await response.json();
    console.log('Data submitted successfully:', result);

  } catch (error) {
    console.error('Error submitting data:', error);
    alert('Error submitting data: ' + error.message);
  }

- Calls the API at the URL stored in pythonURI/api/firstPlaceLeaderboard to retrieve data.
- If the request is successful (response.json()), it logs the fetched data.
- If there's an error (e.g., network issue), it catches and logs the error.
- Perform an HTTP GET request to fetch data.
- Convert the response to JSON format.
- Log the data or catch any errors.

### Discuss the return/response from the method with Algorithm (fetch) and how you handle data.


In [ ]:
if (response.ok) {
    processData(response.json());
  } else {
    handleError(response.statusText);
  }

- If the responses is ok the data is processed and shown in JSON format

### Show how changing data or method triggers a different response, specifically normal conditions and error conditions.

- Not being logged in doesn't allow you to create users.